# Generate GLOBAL DBOF — Combined Script

A single CLI command (`generate-global-combined`) and a single config
file (`combined_global.yaml`) are used.  The **`--subset`** argument selects
which group of properties to compute:

| `--subset` | Fields computed | Output zarr |
|---|---|---|
| `native_fields` | Raw model state variables: `Theta`, `Salt`, `Eta`, `U`, `V`, `W` | `native_fields.zarr` |
| `frontal_structure` | `gradsalt2`, `gradtheta2`, `gradeta2`, `turner_angle` | `frontal_structure.zarr` |
| `kinematic` | `relative_vorticity`, `strain_n`, `strain_s`, `strain_mag`, `divergence`, `coriolis_f`, `rossby_number`, `okubo_weiss` | `kinematic.zarr` |
| `frontogenesis` | `frontogenesis_tendency`,`ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo` | `frontogenesis.zarr` |

---

### Raw model fields (native_fields) are NOT auto-saved

Raw model fields are a dedicated subset (`native_fields`) that you run
explicitly.  All other subsets (`kinematic`, `frontogenesis`, etc.) store only
their computed channels.

---

### All subsets share the same S3 directory via a shared `run_id`

Each subset writes to a different zarr file **inside the same S3 directory**:
```
s3://{bucket}/{folder}/{run_id}/native_fields.zarr
s3://{bucket}/{folder}/{run_id}/kinematic.zarr
s3://{bucket}/{folder}/{run_id}/frontogenesis.zarr
s3://{bucket}/{folder}/{run_id}/frontal_structure.zarr
```

Generate **one shared `run_id`** at the top of the session and reuse it for
every `--subset` invocation below.  This keeps all subsets for a given time
window logically grouped under one directory.

In [3]:
import datetime

# Initial set up
In the future we will want to update this to use docker or conda for end users

## These steps help you build the project on a local machine.
If you are using nrp Jupyterhub, it is recommended you use the next block instead.

#### Build the project
- `pip install .`

**One-time setup:** add the following line to the `[project.scripts]` section
of `pyproject.toml`, then re-run `pip install -e .`:
```toml
generate-global-combined-dataset = "dbof.cli.generate_combined_global:main"
```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the s3 bucket.

Example for installing on linux :
- `sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"`
- `sudo unzip awscliv2.zip`
- `sudo ./aws/install`

## Running in NRP Jupyterhub

This is for running this notebook on nrp Jupyterhub.

This block simply builds the project and installs dependencies not already present on nrp jupyterhub.
You can safely ignore pip warnings

For serious projects, users should use conda or docker but this notebook is meant to be very simple and user friendly.

In [ ]:
#Set True if running on NRP Jupyterhub
RUNNING_ON_NRP = False


if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm
    %pip install aiobotocore

# NOTE IF on NRP Jupyterhub, you will likely need to restart the kernel after running this block

## Set AWS credentials
If you are accessing or writing data to S3, you must set credentials.
NOTE : S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- `$env:AWS_ACCESS_KEY_ID="..."`
- `$env:AWS_SECRET_ACCESS_KEY="..."`

unix:

- `export AWS_ACCESS_KEY_ID=...`
- `export AWS_SECRET_ACCESS_KEY=...`

# Dataset generation config (quick reference)

This job is fully controlled by `configs/combined_global.yaml`.  The config
defines **what time range is scanned**, **how often snapshots are taken**, and
**which property subset to compute**.

### Subset selection
- `active_subset` in the YAML sets the default subset.
- `--subset` on the CLI always takes precedence.
- Each subset has its own `compute_features_channels`, `model_data_feature_channels`,
  and `dataset_name` defined under the `subsets:` key in the config.
- `gradb2` is **always** appended automatically — do not add it to any channel list.

### Dask note for `geostrophic`
The `geostrophic` subset merges two large lazy lineages (velocity Jacobian
gradients + Eta/buoyancy tracer gradients).  The script mitigates this with a
single `dask.compute()` call that fuses and materialises all selected fields
before the zarr write.  If you still see run_spec scheduler warnings, try
reducing `runtime.zarr_async_concurrency` in the config (e.g. to 32 or 64).

### Temporal sampling
- `data.timestep_hours`
  Total time window (in hours) to scan starting from `start_record`.
  Example: `8760` = one LLC model year (336 days).

- `data.sampling_step`
  Spacing **in hours** between snapshots within the window.
  Example: `2190` with `timestep_hours=8760` → 4 evenly spaced snapshots.

- `data.start_record`
  First valid wind/forcing record (default: `1180`). You probably don't want to change this.

Each snapshot corresponds to a **single instantaneous model timestep** (not an average).

### Output / logging
- `output.bucket`, `output.folder`
  S3 location for dataset output.

- `run.run_id`
  Unique identifier for this session.  All subsets run with the same `run_id`
  land in the same S3 directory under different zarr names.

- `run.log_dir`
  Local directory where logs are written.  All subset runs with the same
  `run_id` append to the same log file (`{log_dir}/{run_id}/generate_global.log`).

### Invariants (not configurable)
Grid topology, model cadence (144 timesteps/hour), and dataset offsets are
fixed LLC4320 constants and are enforced in code.

### Examples
See existing configs in `configs/` for examples

# A note about logs
The run logs will be stored locally on your machine in the directory you specify.
- `log_dir/run_id/`

Under the current logic, if you attempt to run the script and the specified
log output path already exists, the script will fail to run.  This is by design.
- `run_id` is also used for the zarr dataset path.  You likely do not want to
  send two different runs to the same dataset.
- You will likely not want to override your previous run logs.

**Exception:** when running multiple subsets with a shared `run_id` (the
intended usage here), the log directory is created once on the first subset
run and reused for subsequent subsets — each appending to the same
`generate_global.log` file.  This is expected behaviour.

To start a completely fresh session, generate a new `run_id` in the cell below.

# Generate a shared run_id for this session

Run this cell **once** at the start of a session.  Reuse `run_id` for every
subset you run below — this groups all output zarr files into the same S3
directory:
```
s3://{bucket}/{folder}/{run_id}/kinematic.zarr
s3://{bucket}/{folder}/{run_id}/frontogenesis.zarr
... etc.
```

In [4]:
#run_id = f"global_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
run_id = "testing_015"  # <-- update if different
print(f"Shared run_id for this session: {run_id}")
print(f"All subsets will write to: s3://dbof/surface_fields/{run_id}/")
print()
print("Update configs/data_access/global.yaml with this run_id when the run is complete.")

Shared run_id for this session: testing_015
All subsets will write to: s3://dbof/surface_fields/testing_015/

Update configs/data_access/global.yaml with this run_id when the run is complete.


---
# Run subsets

Run the cells below **one at a time** (or comment out subsets you don't need).
All cells reuse the `run_id` generated above.

**`frontal_structure` note:** this subset requires `grad_salt2`, `grad_theta2`,
`grad_eta2`, and `turner_angle` to be present in `calculate_additional_fields.py`
before running.

All Dask logs are warnings — do not be alarmed.

## `native_fields`
Raw model state variables only: `Theta`, `Salt`, `Eta`, `U`, `V`, `W` + `gradb2`.

In [6]:
!generate-global \
    --config ../../configs/global.yaml \
    --subset native_fields \
    --run_id $run_id

2026-05-20 16:14:44,513 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/test_logs/testing_015/generate_global.log
2026-05-20 16:14:44,513 | INFO | Arguments parsed successfully. Logging set up. Running script.
2026-05-20 16:14:44,948 | INFO | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-05-20 16:14:44,966 | INFO | State start
2026-05-20 16:14:44,966 | INFO | Found stale lock file and directory '/tmp/dask-scratch-space/scheduler-6zaw7itq', purging
2026-05-20 16:14:44,970 | INFO |   Scheduler at:     tcp://127.0.0.1:39267
2026-05-20 16:14:44,971 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-05-20 16:14:44,971 | INFO | Registering Worker plugin shuffle
2026-05-20 16:14:44,989 | INFO |         Start Nanny at: 'tcp://127.0.0.1:46685'
2026-05-20 16:14:44,994 | INFO |         Start Nanny at: 'tcp://127.0.0.1:45673'
2026-05-20 16:14:44,997 | INFO |         Start Na

## `frontal_structure`
`gradsalt2`, `gradtheta2`, `gradeta2`, `gradb2`.

In [7]:
!generate-global \
    --config ../../configs/global.yaml \
    --subset frontal_structure \
    --run_id {run_id}

2026-05-20 16:25:09,440 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/test_logs/testing_015/generate_global.log
2026-05-20 16:25:09,441 | INFO | Arguments parsed successfully. Logging set up. Running script.
2026-05-20 16:25:09,801 | INFO | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-05-20 16:25:09,814 | INFO | State start
2026-05-20 16:25:09,817 | INFO |   Scheduler at:     tcp://127.0.0.1:44421
2026-05-20 16:25:09,818 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-05-20 16:25:09,818 | INFO | Registering Worker plugin shuffle
2026-05-20 16:25:09,830 | INFO |         Start Nanny at: 'tcp://127.0.0.1:39625'
2026-05-20 16:25:09,833 | INFO |         Start Nanny at: 'tcp://127.0.0.1:39413'
2026-05-20 16:25:09,836 | INFO |         Start Nanny at: 'tcp://127.0.0.1:40589'
2026-05-20 16:25:09,839 | INFO |         Start Nanny at: 'tcp://127.0.0.1:35315'
2026-05-20

## `kinematic`
Velocity-derived scalar fields from a single Jacobian pass.

In [8]:
!generate-global \
    --config ../../configs/global.yaml \
    --subset kinematic \
    --run_id {run_id}

2026-05-20 16:31:03,066 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/test_logs/testing_015/generate_global.log
2026-05-20 16:31:03,066 | INFO | Arguments parsed successfully. Logging set up. Running script.
2026-05-20 16:31:03,430 | INFO | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-05-20 16:31:03,444 | INFO | State start
2026-05-20 16:31:03,447 | INFO |   Scheduler at:     tcp://127.0.0.1:43319
2026-05-20 16:31:03,447 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-05-20 16:31:03,447 | INFO | Registering Worker plugin shuffle
2026-05-20 16:31:03,459 | INFO |         Start Nanny at: 'tcp://127.0.0.1:37149'
2026-05-20 16:31:03,462 | INFO |         Start Nanny at: 'tcp://127.0.0.1:35679'
2026-05-20 16:31:03,465 | INFO |         Start Nanny at: 'tcp://127.0.0.1:46397'
2026-05-20 16:31:03,467 | INFO |         Start Nanny at: 'tcp://127.0.0.1:34065'
2026-05-20

## `frontogenesis`
`frontogenesis_tendency`,`ug`,`vg`,`geostrophic_frontogenesis_tendency`,`ageostrophic_frontogenesis_tendency`

If you see `run_spec` Dask scheduler warnings during this subset, try reducing
`runtime.zarr_async_concurrency` in `combined_global.yaml` (e.g. to 32 or 64).

In [9]:
!generate-global \
    --config ../../configs/global.yaml \
    --subset frontogenesis \
    --run_id {run_id}

2026-05-20 16:38:12,334 | INFO | Log file: /home/lhoffma2/git/llc4320-native-grid-preprocessing/notebooks/test_logs/testing_015/generate_global.log
2026-05-20 16:38:12,334 | INFO | Arguments parsed successfully. Logging set up. Running script.
2026-05-20 16:38:12,826 | INFO | To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-05-20 16:38:12,839 | INFO | State start
2026-05-20 16:38:12,843 | INFO |   Scheduler at:     tcp://127.0.0.1:38215
2026-05-20 16:38:12,843 | INFO |   dashboard at:  http://127.0.0.1:8787/status
2026-05-20 16:38:12,843 | INFO | Registering Worker plugin shuffle
2026-05-20 16:38:12,855 | INFO |         Start Nanny at: 'tcp://127.0.0.1:40055'
2026-05-20 16:38:12,858 | INFO |         Start Nanny at: 'tcp://127.0.0.1:35271'
2026-05-20 16:38:12,860 | INFO |         Start Nanny at: 'tcp://127.0.0.1:45935'
2026-05-20 16:38:12,864 | INFO |         Start Nanny at: 'tcp://127.0.0.1:40767'
2026-05-20

---
# S3 management

In [8]:
# aws cli commands for listing or deleting data in the s3 bucket
# Replace {run_id} with your run_id value, or use the f-string version below.

# List all zarr stores for this run:
# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://dbof/native_grid_dbof_training_data/{run_id}/ --human-readable

# Delete an individual subset zarr (dry-run first):
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://dbof/native_grid_dbof_training_data/{run_id}/kinematic.zarr --recursive --dryrun

# Delete the entire run directory (dry-run first):
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://dbof/native_grid_dbof_training_data/{run_id}/ --recursive --dryrun

